Visualise model output from CF registry data vs baseline ppFEV1

In [4]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import cfr.cfr_viz_helpers as vh

In [ ]:
# Load AC with inferred from 2023 data, 2nd day = 2019 data

df_meas = bd.load_meas_from_excel(
    "CF_Registry_19_23_processed_with_idx", study_folder="CFR"
)

# EXCEL
# 2 entries means there is a 2019 entry for every 2023 entry
# df_res = bd.load_meas_from_excel(
#     "infer_AR_using_19_23_data_2entries_fev1_10122025",
#     # "infer_AR_using_19_23_data_2entries_fev1_fef2575_10122025",
#     study_folder="CFR",
#     str_cols_to_arrays=["Airway resistance (%)"],
# )
# df_res = df_res.drop(columns=["Healthy FEV1 (L)"])

# CSV
# df_res = (
#     bd.load_meas_from_excel(
#         # "infer_AR_using_two_days_model_19_23_data_2entries_fev1_10122025",
#         "infer_AR_using_two_days_model_19_23_data_2entries_fev1_fef2575_10122025",
#         study_folder="CFR",
#         str_cols_to_arrays=["Airway resistance (%)"],
#         use_csv=True,
#         date_cols=["Day"],
#         bypass_sanity_checks=True,
#     )
#     .drop(columns=["Healthy FEV1 (L)"])
#     .rename(columns={"Day": "Date Recorded"})
# )

# Merging
df = df_res.merge(df_meas, on=["ID", "Date Recorded"])

In [5]:
# Load AC from 2019 data with 2nd day = best FEV1 (no FEF2575)
# df = bd.load_meas_from_excel("AR_19_data_with_best_FEV1", study_folder="CFR", str_cols_to_arrays=["Airway resistance (%)"])
df = bd.load_meas_from_excel(
    "infer_all_19_data_with_best_FEV1",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(FEV1|HFEV1_pers)",
        "P(HFEV1|FEF2575, bFEV1)",
    ],
)
print(f"Shape: {df.shape}")

Shape: (2037, 22)


In [29]:
# Process

# Keep only values from 2023
# df23 = df[df["Date Recorded"] == datetime.date(2023, 1, 1)]

df23 = df

AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, {"type": "uniform"})
AC = mh.VariableNode("Airway conductance (%)", 10, 100, 2, {"type": "uniform"})

df23[AC.name] = df23[AR.name].apply(lambda arr: arr[::-1])
df23["ecFEV1 % Predicted (clipped)"] = df23["ecFEV1 % Predicted"].clip(upper=100)
df23["P(ppFEV1|AC)"] = vh.calc_P_ppFEV1_given_AC(df23, AC)
df["P(ppFEV1|AC) ratioed"] = vh.calc_P_ppFEV1_given_AC(df, AC, corr=True)

# Airway conductance

In [ ]:
## FILL ##
ratioed = False
prctile = 25
prctile = 100

df_to_plot, t = vh.filter_confidently_disagreeing_examples(df, prctile, ratioed)
# df_to_plot = df23[df23["P(ppFEV1|AC)"] <= t]

# title = f"Dumbell plot for CF Registry 2023, 2019 2nd day, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, 2019 2nd day, FEV1 & FEF25-75 (2entries), {t*100:.2f}% conf. disagreeing"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1 (no FEF25-75)"
title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, P(ppFEV1|AC ratioed) {prctile:.0f}th prctile"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 (2entries)"
# title = f"Dumbell plot for CF Registry data 2023, FEV1 & FEF2575 (2entries)"

ac_col = AC.name

fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)

# ppfev1_row = "ecFEV1 % Predicted (clipped)"
ppfev1_row = "ecFEV1 % Predicted"

df_to_plot, _, _ = vh.get_dumbell_plot_data(
    df_to_plot, AC.name, AC, ppfev1_row=ppfev1_row
)

# Split dataframe between mild, moderate and severe CF lung disease
# Equivalent to Mean AR_ecFEV1% < 30%, 30 to 60 and > 60%
# Get unique IDs and their corresponding Mean AR_ecFEV1% values
mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
# mask_ppfev1 = df_to_plot["measure"] == f"Mean {ac_col} prediction"

id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"

# Print the sizes to verify
print(f"# Mild: {len(mild_ids)}")
print(f"# Moderate: {len(moderate_ids)}")
print(f"# Severe: {len(severe_ids)}")

# Plot the three groups
vh.plot_dumbell_for_df(
    fig, df_mild, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 3
)
vh.plot_dumbell_for_df(
    fig, df_moderate, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 2
)
vh.plot_dumbell_for_df(
    fig, df_severe, [f"{ac_col} dist", f"{ac_col} mean", ppfev1_row], 1
)


fig.update_layout(
    height=1000,
    # height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="Airway conductance (%)",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(
    # f"{dh.get_path_to_main()}PlotsBreathe/Dumbell_plot_AR_ecFEV1_by_severity/{title}_clipped.pdf"
    # f"{dh.get_path_to_main()}PlotsCFR/{title}_clipped.pdf"
    f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf"
)
# fig.show()

# On this plot, if a person is very sick, it's best fev1 measurement will contain a lot of inflammatory markers (sputum, airway wall inflammation).
# The sicker the person,the more underestimated the AR pred is because the maximum FEV1 blown is not healthy.
# Let's add the FEF25-75.

# 60% of data falls within red band

# TODO: add vertical lines corresponding to key AR values

# Longitudinal AR profile on the web app

# Mild: 890
# Moderate: 793
# Severe: 354
11.988144416876157


In [39]:
df23.head()

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,P(FEV1|HFEV1_pers),P(FEV1_obs|FEV1),Airway resistance (%),mean P(FEV1|HFEV1_pers),FEV1%PersPred,ppFEV1 - pppFEV1,Airway conductance (%),ecFEV1 % Predicted (clipped),P(ppFEV1|AC),P(ppFEV1|AC) ratioed
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,...,"[1.92074181e-247, 2.64360202e-224, 5.15354678e...",4.538045e-06,"[8.65509846e-09, 8.15554203e-08, 6.34526903e-0...",3.071715,48.832655,-1.348417,"[1.58215462e-103, 4.6856718e-81, 2.19166211e-5...",47.484238,0.126144,0.930585
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2.304611e-03,"[2.41434208e-05, 0.000127708133, 0.00052078682...",3.829591,69.720232,-1.906323,"[0.0, 0.0, 7.750625e-318, 1.38091792e-280, 4.1...",67.813909,0.104233,0.956143
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",9.030564e-03,"[0.00883296645, 0.0415722854, 0.0928117744, 0....",5.346436,90.153525,3.588764,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",93.742289,0.121419,0.860508
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,...,"[2.94013653e-226, 4.04664012e-203, 7.88868712e...",6.086227e-05,"[3.27907617e-07, 1.74799278e-06, 8.00925095e-0...",2.580461,55.803975,-1.542358,"[1.96936548e-97, 1.20678788e-76, 3.13909794e-5...",54.261617,0.119938,0.920158
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,...,"[2.19158035e-212, 3.0163691e-189, 5.88023431e-...",1.627400e-16,"[3.83439462e-19, 3.77262211e-18, 5.60801416e-1...",3.152467,29.183496,-0.826397,"[1.58830504e-19, 3.95476136e-13, 7.40140632e-0...",28.357098,0.215366,1.000000


# FEV1 % personalised predicted (Non saturating)

In [6]:
ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
df["mean P(FEV1|HFEV1_pers)"] = df.apply(
    lambda row: ecFEV1.get_mean(row["P(FEV1|HFEV1_pers)"]), axis=1
)

df["FEV1%PersPred"] = df["FEV1"] / df["mean P(FEV1|HFEV1_pers)"] * 100

In [7]:
df

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,best FEV1 old,idx FEV1,idx FEF2575%FEV1,idx best FEV1,"P(HFEV1|FEF2575, bFEV1)",P(FEV1|HFEV1_pers),P(FEV1_obs|FEV1),Airway resistance (%),mean P(FEV1|HFEV1_pers),FEV1%PersPred
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,...,1.64,30,15,32,"[4.15622413e-55, 4.25573467e-47, 8.16549827e-4...","[1.92074181e-247, 2.64360202e-224, 5.15354678e...",4.538045e-06,"[8.65509846e-09, 8.15554203e-08, 6.34526903e-0...",3.071715,48.832655
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,...,2.67,53,18,53,"[1.59171718e-300, 1.47105313e-282, 3.71274261e...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2.304611e-03,"[2.41434208e-05, 0.000127708133, 0.00052078682...",3.829591,69.720232
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,...,4.99,96,36,99,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",9.030564e-03,"[0.00883296645, 0.0415722854, 0.0928117744, 0....",5.346436,90.153525
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,...,1.45,28,20,29,"[6.36205572e-34, 7.93927118e-28, 1.7543138e-22...","[2.94013653e-226, 4.04664012e-203, 7.88868712e...",6.086227e-05,"[3.27907617e-07, 1.74799278e-06, 8.00925095e-0...",2.580461,55.803975
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,...,1.33,18,18,26,"[4.74228193e-20, 4.50605124e-16, 7.54203306e-1...","[2.19158035e-212, 3.0163691e-189, 5.88023431e-...",1.627400e-16,"[3.83439462e-19, 3.77262211e-18, 5.60801416e-1...",3.152467,29.183496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2032,C222738,59,158,1.66,1.18,1.95,Female,2019-01-01,1.66,1.18,...,1.95,33,35,39,"[1.94090843e-116, 1.70532955e-104, 3.19480452e...","[8.96964129e-309, 1.23453147e-285, 2.40664711e...",6.302266e-07,"[1.22721993e-07, 1.78253146e-06, 1.55334891e-0...",2.395400,69.299485
2033,C222739,69,153,1.63,0.66,1.63,Female,2019-01-01,1.63,0.66,...,1.63,32,20,32,"[8.73982189e-51, 7.39906078e-43, 1.16257435e-3...","[4.03898845e-243, 5.55903869e-220, 1.08370193e...",3.401913e-02,"[0.000650025554, 0.00257725852, 0.00834551072,...",1.993599,81.761685
2034,C222741,33,162,1.71,0.86,1.77,Female,2019-01-01,1.71,0.86,...,1.77,34,25,35,"[6.14417333e-80, 3.95741136e-70, 5.05862198e-6...","[2.83944518e-272, 3.90805416e-249, 7.61852194e...",6.743096e-05,"[1.4230942e-06, 5.93297735e-06, 2.06976843e-05...",3.055764,55.959821
2035,C222780,27,176,3.54,4.15,3.54,Female,2019-01-01,3.54,4.15,...,3.54,70,58,70,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3.671734e-02,"[0.0892538366, 0.118583373, 0.123432006, 0.122...",3.962048,89.347723


In [16]:
print(ecFEV1.get_mean(df.loc[1,'P(FEV1|HFEV1_pers)']))
print(df.loc[1,'FEV1'])
print(df.loc[1,'FEV1%PersPred'])
print(df.loc[1, "FEV1 % Predicted"])

3.8295914928424653
2.670000076293945
69.72023207394822
67.81390884010887


In [17]:
import plotly.graph_objects as go
def plot_dumbell_for_df_model_ppfev1(fig, df, measures, col):
    ac_mean = measures[0]
    baseline = measures[1]

    mask = df["measure"] == ac_mean
    fig.add_trace(
        go.Scatter(
            x=df[mask]["value"],
            y=df[mask]["ID"],
            mode="markers",
            marker=dict(color="red", size=4),
            name="ecFEV1 % pers. pred."
        ),
        row=1,
        col=col,
    )

    mask = df["measure"] == baseline
    ecfev1_prct_pred = df[mask]["value"]
    # Where above 100, set to 100
    # ecfev1_prct_pred = np.clip(ecfev1_prct_pred, 0, 100)
    fig.add_trace(
        go.Scatter(
            x=ecfev1_prct_pred,
            y=df[mask]["ID"],
            mode="markers",
            name="ecFEV1 % predicted",
            marker=dict(size=4, color="blue"),
        ),
        row=1,
        col=col,
    )

In [18]:
def get_dumbell_plot_data_model_ppfev1(df, ac_row, ppfev1_row="ecFEV1 % Predicted"):
    # Avoid modifying the original dataframe
    df_res = df.copy()

    ids_sorted = df_res.sort_values("ppFEV1 - pppFEV1", ascending=False)["ID"].values

    df_melted = (
        df_res.melt(
            id_vars=["ID"],
            value_vars=[ppfev1_row, ac_row],
            var_name="measure",
            value_name="value",
        )
        .set_index("ID")
        .loc[ids_sorted]
        .reset_index()
    )

    return df_melted, df_res, ids_sorted

In [24]:
## FILL ##
prctile = 75
prctile = 0

title = f"Dumbell plot for CF Registry data, 2019 with best FEV1, FEV1%PersPred {prctile:.0f}th prctile"
ac_col = "FEV1%PersPred"
ppfev1_row = "ecFEV1 % Predicted"

df["ppFEV1 - pppFEV1"] = df["FEV1 % Predicted"] - df["FEV1%PersPred"]
col = "ppFEV1 - pppFEV1"
t = df[col].abs().quantile(prctile / 100)
df_to_plot = df[df[col].abs() > t]


df_to_plot, _, _ = get_dumbell_plot_data_model_ppfev1(
    df_to_plot, ac_col, ppfev1_row
)

mask_ppfev1 = df_to_plot["measure"] == ppfev1_row
id_ppfev1 = df_to_plot[mask_ppfev1].groupby("ID")["value"].mean()

# Split IDs into three groups based on ecFEV1% values
mild_ids = id_ppfev1[id_ppfev1 >= 70].index
moderate_ids = id_ppfev1[(id_ppfev1 >= 40) & (id_ppfev1 < 70)].index
severe_ids = id_ppfev1[id_ppfev1 < 40].index

# Create the three dataframes
df_mild = df_to_plot[df_to_plot["ID"].isin(mild_ids)]
df_moderate = df_to_plot[df_to_plot["ID"].isin(moderate_ids)]
df_severe = df_to_plot[df_to_plot["ID"].isin(severe_ids)]

# Plot the three groups
fig = make_subplots(
    1, 3, horizontal_spacing=0, column_titles=["Severe CF", "Moderate CF", "Mild CF"]
)
title = f"{title}<br>(#S {len(severe_ids)}, #M {len(moderate_ids)}, #M {len(mild_ids)})"
plot_dumbell_for_df_model_ppfev1(fig, df_mild, [ac_col, ppfev1_row], 3)
plot_dumbell_for_df_model_ppfev1(fig, df_moderate, [ac_col, ppfev1_row], 2)
plot_dumbell_for_df_model_ppfev1(fig, df_severe, [ac_col, ppfev1_row], 1)

fig.update_layout(
    # height=1000,
    height=1800,
    width=1200,
    font=dict(size=14),
    showlegend=True,
    title=title,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
print(min(df_to_plot["value"]))
fig.update_xaxes(
    range=[-25, 160],
    tickvals=[0, 40, 70, 100],
    title="FEV1 % predicted",
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    gridcolor="#2a3f5f",
    zeroline=True,
    zerolinecolor="#2a3f5f",
    zerolinewidth=2,
)
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")
# fig.show()

13.3130788065074


# Does rank change between AC and FEV1%PersPred?

In [49]:
df["ppFEV1 - AC mean"] = df["ecFEV1 % Predicted"] - df[AC.name].apply(AC.get_mean)

In [52]:
# Issue: pppFEV1 is based on AR being 0-2%, not 0%!! →  means pppFEV1 might be lower than AC mean. Hence diff < 0

# How does ppFEV1 metrics compare to AC mean?
df['diff pppFEV1 - diff AC mean'] = df["ppFEV1 - AC mean"] - df['ppFEV1 - pppFEV1']

In [62]:
df = df.sort_values(by=['diff pppFEV1 - diff AC mean'], ascending=False)

In [67]:
df[df['diff pppFEV1 - diff AC mean'].abs() > 6][['ID', 'Age', 'Height', 'Sex', 'FEV1', 'FEF2575', 'best FEV1', 'FEV1 % Predicted', 'diff pppFEV1 - diff AC mean', 'ppFEV1 - AC mean', 'ppFEV1 - pppFEV1']]

,ID,Age,Height,Sex,FEV1,FEF2575,best FEV1,FEV1 % Predicted,diff pppFEV1 - diff AC mean,ppFEV1 - AC mean,ppFEV1 - pppFEV1
2033,C222739,69,153,Female,1.63,0.66,1.63,82.817413,6.804230,7.859958,1.055727
1540,B165718,57,162,Female,2.17,0.76,2.17,84.694427,6.511290,7.589241,1.077951
1950,B172185,73,176,Male,2.34,1.02,2.34,76.985285,6.229667,5.671110,-0.558557
1485,B164954,52,172,Female,2.68,0.97,2.73,87.582749,6.002467,8.284703,2.282236
279,B157577,21,159,Female,2.45,2.98,2.45,77.200572,-6.075926,-7.251525,-1.175599
1624,B166717,29,162,Female,1.69,1.72,1.80,52.755536,-6.111614,-7.592996,-1.481382
998,B162253,20,158,Female,2.60,4.05,2.60,82.937739,-6.487250,-6.631369,-0.144119
176,B156968,29,159,Female,2.32,2.96,2.41,75.350849,-6.658619,-7.681003,-1.022384
1600,B166359,20,162,Female,2.41,2.89,2.47,72.906318,-6.664003,-8.120118,-1.456114
1974,C219729,21,168,Female,2.11,2.25,2.17,59.158449,-6.734589,-8.358186,-1.623597


In [ ]:
# df.to_excel(
#     dh.get_path_to_main() + "ExcelFiles/CFR/dumbel_plots.xlsx",
#     index=False,
# )